In [2]:
# Load the cleaned skincare dataset and check its rows, columns, and first 5 products.
import pandas as pd

df = pd.read_csv("skincare_clean.csv")

print("Total Products:", df.shape[0])
print("Total Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Products:")
display(df.head())


Total Products: 2724
Total Columns: 37

Column Names:
['brand', 'name', 'category', 'ingredients_clean', 'ingredient_count', 'price_usd', 'price_pkr', 'price_capped', 'tag_dry', 'tag_oily', 'tag_combination', 'tag_normal', 'tag_sensitive', 'tag_acne_prone', 'n_tags', 'specificity', 'is_general', 'has_skin_label', 'has_niacinamide', 'has_salicylic_acid', 'has_zinc', 'has_clay', 'has_heavy_oil_butter', 'has_ceramide', 'has_hyaluronic', 'has_squalane', 'has_fragrance', 'has_alcohol_denat', 'has_essential_oil', 'oil_free', 'helps_oil_control', 'helps_acne', 'helps_dryness', 'helps_redness', 'helps_with', 'helps_with_source', 'source']

First 5 Products:


,brand,name,category,ingredients_clean,ingredient_count,price_usd,price_pkr,price_capped,tag_dry,tag_oily,...,has_alcohol_denat,has_essential_oil,oil_free,helps_oil_control,helps_acne,helps_dryness,helps_redness,helps_with,helps_with_source,source
0,The Ordinary,Balancing & Clarifying Serum,Serum,water|lactococcus ferment lysate|propanediol|p...,24,23.9,6624.0,23.9,1.0,1.0,...,0,0,0,0,1,1,0,Signs of Congestion; Dryness; Textural Irregul...,site,brand site
1,The Ordinary,GF 15% Solution,Serum,water|butylene glycol|nicotiana benthamiana he...,12,18.6,5155.0,18.6,1.0,1.0,...,0,0,0,0,0,0,0,Advanced Signs of Aging,site,brand site
2,The Ordinary,Multi-Antioxidant Radiance Serum,Treatment,water|propanediol|ascorbyl glucoside|ergothion...,21,16.8,4656.0,16.8,1.0,1.0,...,0,0,0,0,0,0,0,Dullness; Antioxidant Support,site,brand site
3,The Ordinary,Natural Moisturizing Factors + PhytoCeramides,Moisturizer,water|caprylic/capric triglyceride|cetyl ethyl...,53,13.0,3603.0,13.0,1.0,0.0,...,0,0,0,0,0,1,0,Dryness,site,brand site
4,The Ordinary,Retinal 0.2% Emulsion,Serum,water|coco-caprylate/caprate|propanediol|penty...,31,17.9,4961.0,17.9,1.0,1.0,...,0,0,0,0,0,0,0,Advanced Signs of Aging; Textural Irregulariti...,site,brand site


In [3]:
# Check labeled products, missing ingredients, and skin-type distribution.
print("Skin Label Availability:")
print(df["has_skin_label"].value_counts())

print("\nMissing Ingredient Lists:")
print(df["ingredients_clean"].isnull().sum())

skin_cols = ["tag_dry", "tag_oily", "tag_combination",
             "tag_normal", "tag_sensitive", "tag_acne_prone"]

labeled_df = df[df["has_skin_label"] == 1]

print("\nProducts for Each Skin Type:")
print(labeled_df[skin_cols].sum())


Skin Label Availability:
has_skin_label
1    1557
0    1167
Name: count, dtype: int64

Missing Ingredient Lists:
0

Products for Each Skin Type:
tag_dry            1047.0
tag_oily            935.0
tag_combination    1209.0
tag_normal         1225.0
tag_sensitive       591.0
tag_acne_prone      397.0
dtype: float64


In [4]:
# Select 11 ingredient features to create a basic numerical dataset.
ingredient_cols = [
    "has_niacinamide",
    "has_salicylic_acid",
    "has_zinc",
    "has_clay",
    "has_heavy_oil_butter",
    "has_ceramide",
    "has_hyaluronic",
    "has_squalane",
    "has_fragrance",
    "has_alcohol_denat",
    "has_essential_oil"
]

X_basic = df[ingredient_cols]

print("Total Products:", X_basic.shape[0])
print("Total Features:", X_basic.shape[1])

display(X_basic.head())


Total Products: 2724
Total Features: 11


,has_niacinamide,has_salicylic_acid,has_zinc,has_clay,has_heavy_oil_butter,has_ceramide,has_hyaluronic,has_squalane,has_fragrance,has_alcohol_denat,has_essential_oil
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# Convert complete ingredient lists into numerical features using TF-IDF.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(token_pattern=r"[^|]+")

X_tfidf = tfidf.fit_transform(df["ingredients_clean"])

print("Total Products:", X_tfidf.shape[0])
print("Total Ingredient Features:", X_tfidf.shape[1])


Total Products: 2724
Total Ingredient Features: 7316


In [6]:
# Display the first 10 ingredient names and their TF-IDF values for the first product.
ingredient_names = tfidf.get_feature_names_out()

print("First 10 Ingredients:")
print(ingredient_names[:10])

print("\nTF-IDF Values for First Product:")
print(X_tfidf[0].toarray()[0][:10])


First 10 Ingredients:
['(-)-alpha-bisabolol' '(bergamot) fruit oil' '(bilberry) extract'
 '(evening primrose) oil' '(f.i.l. : b257052/2)' '(fragrance)'
 '(guar) gum' '(jojoba) seed oil' '(mel) honey'
 '(microcrystalline cellulose']

TF-IDF Values for First Product:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [7]:
# Display the ingredients present in the first product with their TF-IDF scores.
first_product = X_tfidf[0].toarray()[0]

print("Ingredients in First Product:")

for i in range(len(first_product)):
    if first_product[i] > 0:
        print(ingredient_names[i], ":", round(first_product[i], 3))


Ingredients in First Product:
acetyl glucosamine : 0.204
adenosine : 0.167
carnitine : 0.33
chlorphenesin : 0.15
citric acid : 0.097
ectoin : 0.271
ethylhexylglycerin : 0.098
glycerin : 0.06
isoceteth-20 : 0.265
lactococcus ferment lysate : 0.259
pentylene glycol : 0.125
phenoxyethanol : 0.074
polylysine : 0.33
potassium azeloyl diglycinate : 0.3
propanediol : 0.106
sarcosine : 0.315
sclerotium gum : 0.203
silanetriol : 0.303
sodium benzoate : 0.111
sodium chloride : 0.152
sodium citrate : 0.142
trisodium ethylenediamine disuccinate : 0.186
water : 0.057
xanthan gum : 0.097


In [8]:
# Save the basic and TF-IDF ingredient features for K-Means clustering.
from scipy.sparse import save_npz
import joblib

X_basic.to_csv("ingredients_basic.csv", index=False)
save_npz("ingredients_tfidf.npz", X_tfidf)
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("Ingredient features saved successfully!")


Ingredient features saved successfully!


### Conclusion

In this phase, the cleaned SkinMatch skincare dataset was loaded and prepared for machine learning.

The dataset contains **2,724 skincare products and 37 columns**, including product details, ingredients, prices, and skin-type labels.

During data inspection, **1,557 products** were found to have skin-type labels, while **1,167 products** were unlabeled. No missing ingredient lists were found.

Two different ingredient feature representations were prepared:

- **Basic features:** 11 numerical ingredient indicators, such as niacinamide, salicylic acid, ceramides, and hyaluronic acid.
- **TF-IDF features:** 7,316 numerical features extracted from the complete ingredient lists.

The TF-IDF method was used to represent individual ingredients numerically while preserving complete ingredient names.

Both feature sets were successfully created and saved for further use.

The prepared features will be used in the next phases to perform **K-Means clustering, KNN classification, and ingredient-based skincare recommendations**.

This phase successfully completed the feature preparation required for the SkinMatch recommendation system.